# Real-time Inference Test

Tests the deployed anomaly detection model via the ModelMesh inference endpoint.
Simulates the same request flow that the IoT consumer (`messaging`) uses.

The model expects **2 features**: `[vibration, temperature]`.

In [ ]:
import requests
import json
import os
import urllib3
urllib3.disable_warnings()

In [ ]:
MODELMESH_URL = os.environ.get(
    'MODELMESH_URL',
    'http://modelmesh-serving.ml-development:8008'
)
INFERENCE_PATH = '/v2/models/inference-service/infer'

print(f"Endpoint: {MODELMESH_URL}{INFERENCE_PATH}")

In [ ]:
def predict_vibration(vibration, temperature=50.0):
    """Send vibration+temperature to ModelMesh and return the prediction."""
    payload = {
        "inputs": [{
            "name": "predict",
            "shape": [1, 2],
            "datatype": "FP64",
            "data": [[vibration, temperature]]
        }]
    }
    try:
        resp = requests.post(
            f"{MODELMESH_URL}{INFERENCE_PATH}",
            json=payload,
            timeout=10,
            verify=False
        )
        if resp.status_code == 200:
            result = resp.json()
            prediction = result['outputs'][0]['data'][0]
            return 'NORMAL' if prediction == 1 else 'ANOMALY'
        else:
            return f'ERROR: {resp.status_code} {resp.text[:200]}'
    except requests.exceptions.ConnectionError:
        return 'ERROR: Cannot connect to ModelMesh (check namespace/service)'
    except Exception as e:
        return f'ERROR: {str(e)}'

## Test with known values

In [ ]:
test_cases = [
    (12.0, 50.0, 'Normal'),
    (15.0, 52.0, 'Normal'),
    (17.5, 48.0, 'High-normal'),
    (25.0, 55.0, 'Elevated'),
    (35.0, 60.0, 'Anomaly spike'),
    (45.0, 75.0, 'Severe anomaly'),
]

print(f"{'Vibration':>10} {'Temp':>6}  {'Expected':>16}  {'Prediction':>12}")
print('-' * 54)
for vib, temp, desc in test_cases:
    result = predict_vibration(vib, temp)
    print(f"{vib:10.1f} {temp:6.1f}  {desc:>16}  {result:>12}")

## Simulate streaming sensor data

In [ ]:
import numpy as np

np.random.seed(42)
n_readings = 20

vibrations = np.concatenate([
    np.random.uniform(10, 18, 15),
    np.random.uniform(30, 50, 5),
])
temperatures = np.random.uniform(45, 55, n_readings)
shuffle_idx = np.random.permutation(n_readings)
vibrations = vibrations[shuffle_idx]
temperatures = temperatures[shuffle_idx]

anomaly_count = 0
for i in range(n_readings):
    result = predict_vibration(vibrations[i], temperatures[i])
    if result == 'ANOMALY':
        anomaly_count += 1
    marker = ' !!' if result == 'ANOMALY' else ''
    print(f"[{i+1:2d}/{n_readings}] vib={vibrations[i]:6.2f} temp={temperatures[i]:5.1f}  ->  {result}{marker}")

print(f"\nSummary: {anomaly_count} anomalies detected in {n_readings} readings")